# Uji tambahan CNN: EfficientNet-B0

Jalankan di **akun Google yang dulu menjalankan `01_efficientnet_b0_colab.ipynb` (R15)**, yaitu akun yang
`MyDrive/TA_R15_hasil/outputs/final_models/` berisi `efficientnet_b0`. Sel 1 memeriksa ini dan berhenti bila akunnya salah.

Notebook ini memakai bundel R15 yang sama dengan run resmi (TA_R15_COLAB) dan folder tambahan TA_R15_UJI_TAMBAHAN.
Cukup **Runtime > Run all** dengan GPU **L4**. Perkiraan total 11-14 jam.

**Aturan yang mengikat:**
- Kode training, data, cache embedding, dan feature store sama dengan run resmi; hash protokol dan metode diperiksa di setiap uji.
- Hasil resmi `MyDrive/TA_R15_hasil` hanya dibaca. Semua hasil uji ditulis ke folder baru `MyDrive/TA_R15_uji_tambahan_hasil/<id uji>/`.
- Hold-out tidak dibaca di notebook ini. Evaluasi dilakukan secara lokal dari checkpoint yang tersimpan.
- Setiap sel aman dijalankan ulang. Uji yang selesai dilewati, training yang terputus dilanjutkan dari `last.pt`.
- Urutan sel = prioritas. Bila runtime terputus, cukup Run all lagi.
- Tidak ada yang dihapus atau ditimpa. Bila masukan suatu folder uji berbeda, sel berhenti dengan pesan.


In [ ]:
# Sel 1: mount Drive, pasang paket versi run resmi, salin bundel R15 + verifikasi sha256 (5-10 menit pada sesi pertama)
ARCH = "efficientnet_b0"
from google.colab import drive
drive.mount("/content/drive")
import sys
from pathlib import Path
MYDRIVE = Path("/content/drive/MyDrive")
CARI = [MYDRIVE, MYDRIVE.parent / ".shortcut-targets-by-id"]   # akun lain: pintasan ke folder bersama
def cari(penanda):
    return next((p.parent for root in CARI for d in range(4) for p in sorted(root.glob("/".join(["*"] * d + [penanda])))), None)
BUNDEL, ADDON = cari("BUNDEL_R15.json"), cari("BUNDEL_UJI_TAMBAHAN.json")
assert BUNDEL, "Folder TA_R15_COLAB tidak ditemukan (akun yang bukan pemilik: tambahkan pintasannya ke My Drive)."
assert ADDON, "Folder TA_R15_UJI_TAMBAHAN tidak ditemukan (akun yang bukan pemilik: tambahkan pintasannya ke My Drive)."
print("Bundel:", BUNDEL, "| Tambahan:", ADDON)
sys.path[:0] = [str(BUNDEL), str(ADDON)]
sys.dont_write_bytecode = True   # jangan menulis __pycache__ ke folder bersama
import colab_setup as C
C.pasang_paket()
import colab_tests_setup as CR
INFO = CR.siapkan(ARCH, BUNDEL, ADDON)


In [ ]:
# Sel 2: lingkungan (harus sama dengan run resmi: torch 2.6.0+cu124, GPU L4)
import platform, torch, json
env = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "cudnn": torch.backends.cudnn.version(), "gpu": torch.cuda.get_device_name(0), **INFO}
assert torch.__version__.startswith("2.6.0") and "L4" in env["gpu"], env
CR.HASIL_UJI.mkdir(parents=True, exist_ok=True)
(CR.HASIL_UJI / f"env_{ARCH}.json").write_text(json.dumps(env, indent=2)); env


## G0L4. Reproduksi model final S2 resmi di L4
Model final S2 dilatih ulang dengan seed dan E\* resmi. Bila checkpoint identik bit per bit dengan checkpoint resmi, lingkungan Colab sekarang sama dengan run resmi.


In [ ]:
CR.jalankan("G0L4", ARCH)


## U2. Leave-source-out AF/VT
Model final S2 tanpa 12 pasien development yang memiliki AF di luar AFDB atau VT di luar VFDB (3.134 window dikeluarkan), seed dan E\* resmi.


In [ ]:
CR.jalankan("U2", ARCH)


## U3. Baseline citra saja (image-only)
Head citra-saja tanpa fitur tabular, GRU RR, dan expert tabular, protokol S2 (backbone beku, tau 2): CV 5 fold, E\* dengan aturan resmi, lalu model final.


In [ ]:
CR.jalankan("U3", ARCH)


## U5. Backbone acak beku (kontrol ImageNet)
Encoder diinisialisasi acak (seed 42), statistik BatchNorm dikalibrasi dari 512 window development tanpa label, lalu cache embedding baru dan protokol S2 penuh: CV 5 fold, E\* dengan aturan resmi, model final. Cache disimpan di disk lokal runtime dan diarsipkan ke folder uji.


In [ ]:
CR.jalankan("U5", ARCH)


## U4. Variasi seed model final S1 dan S2
Model final S1 dan S2 dengan seed final lain (1042 dan 2042), E\* resmi.


In [ ]:
CR.jalankan("U4", ARCH, "S1", 1042)
CR.jalankan("U4", ARCH, "S2", 1042)
CR.jalankan("U4", ARCH, "S1", 2042)
CR.jalankan("U4", ARCH, "S2", 2042)


## U4 (S3). Variasi seed fine-tuning S3
Fine-tuning penuh resep S3 resmi (E\* S3 resmi) dari model final U4 S2 seed yang sama, sehingga rantai S2 ke S3 diulang utuh per seed. Harus setelah sel U4 di atas.


In [ ]:
CR.jalankan("U4", ARCH, "S3", 1042)
CR.jalankan("U4", ARCH, "S3", 2042)


## U6. Learning curve tingkat pasien
Model final S2 pada 25/50/75 % pasien development, berstrata per kelas, 3 ulangan bersarang, E\* dan seed resmi (9 run singkat).


In [ ]:
CR.jalankan("U6", ARCH, fraksi=0.25, ulang=0)
CR.jalankan("U6", ARCH, fraksi=0.5, ulang=0)
CR.jalankan("U6", ARCH, fraksi=0.75, ulang=0)
CR.jalankan("U6", ARCH, fraksi=0.25, ulang=1)
CR.jalankan("U6", ARCH, fraksi=0.5, ulang=1)
CR.jalankan("U6", ARCH, fraksi=0.75, ulang=1)
CR.jalankan("U6", ARCH, fraksi=0.25, ulang=2)
CR.jalankan("U6", ARCH, fraksi=0.5, ulang=2)
CR.jalankan("U6", ARCH, fraksi=0.75, ulang=2)


## Status
Semua baris harus `selesai`. Setelah itu unduh folder `MyDrive/TA_R15_uji_tambahan_hasil` (tanpa subfolder `feature_store`) ke komputer lokal.


In [ ]:
import pandas as pd
pd.DataFrame(CR.status(ARCH))
